**MATH70148 - Probabilistic Generative Models 2025-2026**

# Coursework 2 - Part II


**Instructions:**

You will implement a minimal version of the GPT architecture and train it to generate synthetic text in the style of the Shakespeare's works. Follow this notebook and complete the missing parts according to the given instructions. Register the code you generate in the parts indicated as `YOUR CODE HERE` and your responses (when applicable) by creating a new cell in this notebook. Your submission will be assessed from this notebook only, **no pdf report is needed**.

**Comment on running time:** It is expected that this model is to be trained over at least 5000 epochs to produce text resembling the stucture of a play, and you are strongly encouraged to run it on Google Colab (where the 5000 epochs should take around 6 minutes if using a T4 runtime type) or a computer with a sufficiently capable GPU. For reference, if ran on a regular MPS-enabled Apple M3 Pro Macbook Pro, this code will take ~10 minutes, and ~5.5 minutes if using a NVIDIA GeForce GTX 1080 GPU.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## 1. Download the corpus and tokenise it

In [4]:
# If using Colab, consider:
#from google.colab import drive
#drive.mount('/content/drive/')

In [5]:
# Load the provided dataset

In [6]:
# Read the entire Shakespeare corpus into memory
# We explicitly set UTF-8 encoding to avoid platform-dependent issues.
with open('shakespeare.txt', 'r', encoding='utf-8') as file:
    text = file.read()

# Basic corpus statistics
num_chars = len(text)

print("=== Corpus Statistics ===")
print(f"Total number of characters: {num_chars:,}")
print(f"First 200 characters preview:\n")
print(text[:200])

=== Corpus Statistics ===
Total number of characters: 1,115,394
First 200 characters preview:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [7]:
# Extract the set of unique characters appearing in the text
# `set(text)` removes duplicates
# `sorted(...)` ensures a deterministic ordering (important for reproducibility)
chars = sorted(list(set(text)))

# Size of the vocabulary (number of unique characters)
vocab_size = len(chars)

# Display results
print("Unique characters (sorted):")
print("".join(chars))  # Join list into a readable string

print("\nVocabulary size:")
print(f"{vocab_size} unique characters")

Unique characters (sorted):

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

Vocabulary size:
65 unique characters


In [8]:
# --------------------------------------------------
# Character-level tokenizer
# --------------------------------------------------

# NOTE:
# The model will operate at the character level, meaning that the tokens are characters (and not words)

# Create a mapping from character to integer index
# This is our "string-to-index" (stoi) dictionary.
stoi = {char: idx for idx, char in enumerate(chars)}

# Create the inverse mapping: index to character
# This is needed to convert model outputs back to text.
itos = {idx: char for idx, char in enumerate(chars)}


def encode(s):
    """
    Convert a string into a list of integer token IDs.

    Args:
        s (str): Input string.

    Returns:
        List[int]: List of corresponding character indices.
    """
    return [stoi[char] for char in s]


def decode(indices):
    """
    Convert a list of integer token IDs back into a string.

    Args:
        indices (List[int]): List of character indices.

    Returns:
        str: Decoded string.
    """
    return ''.join(itos[idx] for idx in indices)


In [9]:
# Convert the entire corpus into a PyTorch tensor of token IDs.
# We use dtype=torch.long because embedding layers expect integer (int64) indices.
data = torch.tensor(encode(text), dtype=torch.long)

# Inspect the shape of the resulting tensor
print("Data tensor shape:", data.shape)

Data tensor shape: torch.Size([1115394])


In [10]:
# Display a short excerpt of the original text and its corresponding
# integer (token ID) representation to illustrate the encoding process.

N = 50

print(f"Text containing the first {N} characters:")
print("-" * 50)
print(text[:N])

print("\n" + "-" * 50)
print("Its representation as a PyTorch tensor:\n")
print(data[:N])

Text containing the first 50 characters:
--------------------------------------------------
First Citizen:
Before we proceed any further, hear

--------------------------------------------------
Its representation as a PyTorch tensor:

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])


## 2. Dataset split

In [11]:
# Split the dataset into training (90%) and validation (10%) sets.
# We consider a sequential split to preserve temporal structure.

n = int(0.9 * len(data))  # 90% for training
train_data = data[:n]
val_data = data[n:]

print(f"Training corpus size: {len(train_data):,} characters "
      f"({train_data.shape[0] / data.shape[0]:.2%} of total)")

print(f"Validation corpus size: {len(val_data):,} characters "
      f"({val_data.shape[0] / data.shape[0]:.2%} of total)")

Training corpus size: 1,003,854 characters (90.00% of total)
Validation corpus size: 111,540 characters (10.00% of total)


### On the data–label mechanics

- We access the data through *context windows*: contiguous blocks of text in numerical (token) representation of size `block_size`.

- The model is self-supervised. That is, we train it so that, given a sequence of tokens ${x}_{i:j}$, it predicts the next token $x_{j+1}$.

- The labels are obtained from the same context window by shifting it with an offset of 1 position.

In [12]:
# EXAMPLE:

# Given a context window of size 7,
# we construct an input–target pair (X, Y),
# where Y is simply X shifted by one position.

block_size = 7

print("A context window (X, Y) looks like:\n")

# Input sequence (context)
X = data[:block_size]
print(f"X (token IDs): {[x.item() for x in X]}")
print(f"--> decode(X): {decode([x.item() for x in X])}")

print("-" * 40)

# Target sequence (shifted by one position)
Y = data[1:block_size + 1]
print(f"Y (token IDs): {[y.item() for y in Y]}")
print(f"--> decode(Y): {decode([y.item() for y in Y])}")

A context window (X, Y) looks like:

X (token IDs): [18, 47, 56, 57, 58, 1, 15]
--> decode(X): First C
----------------------------------------
Y (token IDs): [47, 56, 57, 58, 1, 15, 47]
--> decode(Y): irst Ci


In [13]:
# EXAMPLE (continued):

# However, within each context window we use the information
# in an autoregressive manner, generating multiple training
# examples from a single block.

x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t + 1]      # Tokens available at time step t
    target = y[t]            # Next token to predict

    print(f"When the input is {context.tolist()}, the target is: {target.item()}")

When the input is [18], the target is: 47
When the input is [18, 47], the target is: 56
When the input is [18, 47, 56], the target is: 57
When the input is [18, 47, 56, 57], the target is: 58
When the input is [18, 47, 56, 57, 58], the target is: 1
When the input is [18, 47, 56, 57, 58, 1], the target is: 15
When the input is [18, 47, 56, 57, 58, 1, 15], the target is: 47


Therefore, each context window generates a number of training observations equal to its length.

We now turn to *batches*, which allows us to process multiple context windows in parallel. The key point is that each context window is independent: there is no computation that occurs across windows, only parallel computation within them. In other words, the autoregressive sequences corresponding to different context windows are not mixed. See the following example.

In [14]:
torch.manual_seed(1337)

batch_size = 4
block_size = 8  # maximum context length considered for prediction
# These hyperparameters will be redefined for the final training phase


def get_batch(split):
    """
    Generate a small batch of input (x) and target (y) sequences.

    Each batch contains `batch_size` independent context windows
    of length `block_size`, sampled randomly from the dataset.
    """
    data_split = train_data if split == 'train' else val_data

    # Random starting positions for each context window
    ix = torch.randint(len(data_split) - block_size, (batch_size,))

    # Construct input and target batches
    x = torch.stack([data_split[i:i + block_size] for i in ix])
    y = torch.stack([data_split[i + 1:i + block_size + 1] for i in ix])

    return x, y


# Sample a training batch
xb, yb = get_batch('train')

print("Inputs:")
print("Shape:", xb.shape)   # (batch_size, block_size)
print(xb)

print("\nTargets:")
print("Shape:", yb.shape)   # (batch_size, block_size)
print(yb)

Inputs:
Shape: torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])

Targets:
Shape: torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


### Questions
 - Q1 (5% marks): Explain why autoregressive language modeling is considered self-supervised. What are the inputs and labels in this setup?
 - Q2 (5% marks): From a context window of size $T$, how many training examples are generated? Explain why.

Q1:

It is considered self-supervised because the supervision signal is derived directly from the data itself. The target tokens $x_t$ already exist in the original text right after $x_{1:t-1}$, so no human labeling is required. 

Given an ordered sequence of tokens:

$$
(x_1, x_2, \dots, x_{k+1})
$$

The input is the sequence of tokens available at the current steps:

$$
x = (x_1, x_2, \dots, x_k)
$$

The label is the same sequence shifted one position forward:

$$
y = (x_2, x_3, \dots, x_{k+1})
$$

Thus, the model is trained to predict the next token (character) $x_{t+1}$ given the previous tokens $x_{1:t}$.

During training, multiple sequences are processed in parallel and stacked into tensors $X,Y \in \mathcal{V} ^{B \times k}$, where $B$ is the batch size, $k$ is the context length, and $\mathcal{V}$ is the vocabulary

Q2: 

$T$ training samples are generated. Using the chain rule of probability, the distribution of an ordered sequence can be factorised as

$$
p(x_{1:T+1}) = \prod_{t=1}^T p(x_{t+1} | x_{1:t})
$$

where the sequence modelling problem reduces to the next-token prediction task. 

Given input sequence $x =x_{1:T}$ and its shifted target sequence $y=x_{2:T+1}$, the model computes a prediction at every time step $t \in \{ 1, \dots, T\}$. For example:

- At $t=1$: input $x_1$ predicts target $x_2$
- At $t=2$: input $x_{1:2}$ predicts target $x_3$
- At $t=T$: input $x_{1:T}$ predicts target $x_{T+1}$

Therefore, a context window of size $T$ produces $T$ training samples.

## 3. Self-attention

- Q3 (15% marks): Implement the missing parts in the attention head class. Recall that in this model you need to implement masked attention.

In [15]:
# Embedding dimension (model "width"): each token is represented as a vector in R^{n_embd}
n_embd = 64
dropout = 0.0


class Head(nn.Module):
    """
    A single self-attention head (causal / autoregressive).

    Given an input sequence of token embeddings x of shape (B, T, n_embd),
    this module produces an output of shape (B, T, head_size) by:
      1) projecting x into queries, keys, and values
      2) computing attention weights via scaled dot-product attention
      3) applying a causal mask so tokens cannot attend to the future
      4) computing a weighted sum of value vectors
    """

    def __init__(self, head_size: int):
        super().__init__()


        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # -------------------------------------------------------- #


        # Causal mask (lower-triangular): prevents attending to future positions.
        # register_buffer makes it part of the module state (moves with .to(device))
        # but it is not a learnable parameter.
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

        # Dropout applied to the attention weights (common regularization in Transformers)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:


        """
        Args:
            x: (B, T, n_embd) input token embeddings

        Returns:
            out: (B, T, head_size) attention output for this head
        """

        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        B, T, n_embd = x.shape

        # 1) projecting x into queries, keys, and values
        # Shape: (B, T, head_size)
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # 2) computing attention weights via scaled dot-product attention
        # Q @ K^T / sqrt(head_size)
        k = k.transpose(-2, -1) # Shape: (B, head_size, T)
        Dk = k.shape[1]
        scores = q @ k * (Dk ** -0.5) # Shape: (B, T, T)

        # 3) applying a causal mask so tokens cannot attend to the future
        # Slice self.tril[:T, :T] so it adapts if the input T < block_size
        # == 0 create true mask for upper-triangular and false mask otherwise
        # The mask forces upper-triangular elements to -infinity
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # 4) computing a weighted sum of value vectors
        weights = F.softmax(scores, dim=-1) # Convert to probabilities for each query row
        weights = self.dropout(weights)
        out = weights @ v # (B, T, T) @ (B, T, head_size) --> (B, T, head_size)
        # -------------------------------------------------------- #

        return out

### Questions
 - Q3 (continued): Explain why the mask is needed.  

The mask is needed to enforce the autoregressive property of the decoder so that the model cannot use information from future tokens when predicting the next token.

The decoder defines a conditional generative model over an output sequence $Y = (y_1, \dots, y_T)$, factorising the joint distribution autoregressively:

$$
p(Y) = \prod_{t=1}^T p(y_t \mid y_{<t})
$$

Thus, the prediction of token $y_t$ must depend only on the previous tokens $y_{<t}$. 

In the decoder, the hidden representation $z_t$ is computed using self-attention as a convex combination of values:

$$
z_t = \sum_{j=1}^{T} \alpha_{tj} v_j
$$

where the attention weights $\alpha_{ij}$ are

$$
\alpha_{tj} = \frac{\exp(s(q_t, k_j))}{\sum_{\ell=1}^T \exp(s(q_t, k_\ell))}
$$

Without a mask, the query $q_t$ could attend to keys $k_j$ corresponding to future tokens $j > t$, resulting attention weight $\alpha_{tj} > 0$. This would allow $z_t$ to use future information, violating the autoregressive factorisation $p(y_t \mid y_{<t})$. 

To prevent this information leakage, a causal mask is applied to set the upper-triangular entries of the similarity scores matrix where $j > t$ to $-\infty$. Applying the softmax we get: 

$$

\exp(-\infty) = 0
$$

so the attention weights for future tokens become zero. As such, each position $t$ can attend only to tokens at positions $j \le t$ , preserving the autoregressive structure of the model.

## 4. Multi-head Attention

- Q4 (10% marks): Implement the missing parts in the multihead attention class.

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multiple self-attention heads running in parallel.

    Each head independently computes attention over the same input.
    Their outputs are concatenated and linearly projected back to the
    embedding dimension (n_embd).
    """

    def __init__(self, num_heads: int, head_size: int):
        super().__init__()

        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        # Instantiate multiple self-attention heads
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

        # Create linear projection for W_O that mixes the contributions of each attention head
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        # -------------------------------------------------------- #


        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T, n_embd)

        Returns:
            out: (B, T, n_embd)
        """
        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        # Compute the forward pass to get the output from each head
        head_outputs = [h(x) for h in self.heads] # list of num_heads tensor of shape: (B, T, head_size)

        # Concatenate the outputs along the feature dimension
        out = torch.cat(head_outputs, dim=-1) # Shape: (B, T, num_heads * head_size)

        # Linearly project the concatenated output back to the embedding dimension
        out = self.proj(out) # Shape: (B, T, n_embd)

        # Apply dropout
        out = self.dropout(out)
        # -------------------------------------------------------- #


        return out

## 5. MLP stage

- Q5 (7.5% marks): Implement the missing parts in the FeedForward attention class.

In [17]:
class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network (single-hidden layer MLP)
    This module is applied independently to each position in the sequence.

    Input:  (B, T, n_embd)
    Output: (B, T, n_embd)
    """

    def __init__(self, n_embd: int):
        super().__init__()

        # The Transformer FFN consists of two linear layers with a non-linearity in between.
        # The inner layer expands the dimensionality by a factor of 4,
        # then projects back to the original embedding size.

        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # Expand dimension by a factor of 4
            nn.ReLU(),                      # Apply non-linear activation function
            nn.Linear(4 * n_embd, n_embd),  # Project back to original embedding size
            nn.Dropout(dropout)             # Regularisation
        )
        # -------------------------------------------------------- #



    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Applies the feed-forward network independently at each time step.
        No interaction occurs across sequence positions.
        """
        return self.net(x)

### Questions
 - Q6 (10% marks): Explain the relationship between the hyperparameters `n_head` and `head_size` in the `MultiHeadAttention` class. Consider their role within the decoder block (i.e., attention + feed-forward).

In multi-head attention, the input embedding of dimension $n_{embd}$ is split across multiple attention heads. The hyperparameter `n_head` controls the number of attention heads while `head_size` controls the dimensionality of the representation used by each head. Together, they determine how the embedding space is partitioned.

For each head $h = 1, \dots, n_{head}$, the input token $x_i$ is projected into a smaller subspace using learnable weights $W_Q^{(h)}, W_K^{(h)}, W_V^{(h)}$:

$$
q_i^{(h)} = W_Q^{(h)} x_i, \quad k_j^{(h)} = W_K^{(h)} x_j, \quad v_j^{(h)} = W_V^{(h)} x_j
$$

Each head performs self-attention independently and produces an output vector of dimension `head_size` . The outputs of all heads are concatenated:

$$
\text{Concat}(\text{head}^{(1)}(q_i), \dots, \text{head}^{(n_{head})}(q_i))
$$

resulting in a vector of dimension

$$
n_{head} \times \text{head\_size}
$$

A linear projection $W_O$ mixes the contributions of each attention head, back to the original embedding dimension:

$$
\text{MHA}(q_i) = W_O \text{Concat}(\text{head}^{(1)}(q_i), \dots, \text{head}^{(n_{head})}(q_i))
$$

To avoid dimension mismatch, the embedding dimension must satisfy:

$$
n_{embd}=n_{head} \times \text{head\_size}
$$

This relationship is important within the decoder block. The MHA layer must output a representation of dimension `n_embd` so that the residual connection can be added:

$$
X_{\text{updated}} = X + \text{MHA}(X)
$$

The resulting representation is then passed to the position-wise feed-forward network, which also expects an input of dimension `n_embd` , allowing another residual connection:

$$
Z_{\text{updated}} = Z + \text{MLP}(Z)
$$

Thus `n_embd` and `head_size` determine how the model distributes representation capacity across multiple attention heads while preserving the embedding dimension required by the decoder architecture.

## 6. Decoder block

- Q7 (10% marks): Implement the missing parts in the DecoderBlock class.

In [ ]:
class DecoderBlock(nn.Module):
    """
    Transformer decoder block: COMMUNICATION followed by COMPUTATION.

    - Communication: masked multi-head self-attention (tokens exchange information)
    - Computation: position-wise feed-forward network (nonlinear transformation per token)

    Your implementation should use the Pre-LayerNorm (Pre-LN) variant:
    LayerNorm is applied before each sublayer, and residual (skip) connections are applied around each sublayer.
    """

    def __init__(self, n_embd: int, n_head: int):
        # n_embd: embedding dimension, n_head: number of attention heads

        super().__init__()

        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        # Determine the size of each attention head
        head_size = n_embd // n_head #

        # Communication: masked multi-head self-attention (tokens exchange information)
        self.sa = MultiHeadAttention(n_head, head_size)

        # Computation: position-wise feed-forward network (nonlinear transformation per token)
        self.ffwd = FeedForward(n_embd)

        # Normalisation layers for the Pre-LN
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        # -------------------------------------------------------- #



    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T, n_embd) input embeddings

        Returns:
            x: (B, T, n_embd) output after attention + feed-forward, with residuals
        """
        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        # LayerNorm -> Attention -> Residual connection
        x = x + self.sa(self.ln1(x))
        # LayerNorm -> Feedforward -> Residual connection
        x = x + self.ffwd(self.ln2(x))
        # -------------------------------------------------------- #



        return x

## 7. GPT Model: Putting it all together

- Q8 (17.5% marks): Implement the missing computation of the loss in the `GPTLanguageModel` class and the `generate` method to sample tokens that autoregressively extend a sequence.

In [ ]:
class GPTLanguageModel(nn.Module):
    """
    A minimal GPT-style (decoder-only) Transformer language model.

    Components:
      - Token embeddings: map token IDs -> vectors in R^{n_embd}
      - Positional embeddings: encode token position within the context window
      - A stack of decoder blocks: masked self-attention + feed-forward (+ residuals)
      - Final layer norm + linear head: produce logits over the vocabulary

    Input:
      idx: (B, T) integer token IDs

    Output:
      logits: (B, T, vocab_size)
      loss: scalar cross-entropy loss (if targets are provided)
    """

    def __init__(self):
        super().__init__()

        # Token and position embeddings
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # Stack of decoder blocks
        self.blocks = nn.Sequential(
            *[DecoderBlock(n_embd, n_head=n_head) for _ in range(n_layer)]
        )

        # Final normalization and language modeling head
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # Weight initialization (close to the GPT/Transformer convention)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        """
        Args:
            idx: (B, T) token IDs
            targets: (B, T) token IDs (optional)

        Returns:
            logits: (B, T, vocab_size)
            loss: scalar loss if targets is provided, else None
        """
        B, T = idx.shape

        # Token embeddings: (B, T, n_embd)
        tok_emb = self.token_embedding_table(idx)

        # Positional embeddings: (T, n_embd)
        # We create positions [0, 1, ..., T-1] on the correct device.
        pos = torch.arange(T, device=idx.device)
        pos_emb = self.position_embedding_table(pos)

        # Add token and positional information (broadcast pos_emb over batch dimension)
        x = tok_emb + pos_emb  # (B, T, n_embd)

        # Decoder blocks + final layer norm
        x = self.blocks(x)     # (B, T, n_embd)
        x = self.ln_f(x)       # (B, T, n_embd)

        # Convert to vocabulary logits
        logits = self.lm_head(x)  # (B, T, vocab_size)

        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        if targets is None: # inference
            loss = None
        else: # training
            B, T, C = logits.shape
            logits = logits.view(B * T, C) 
            targets = targets.view(B * T) 
            loss = F.cross_entropy(logits, targets)
        # -------------------------------------------------------- #



        return logits, loss

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0):
        """
        Autoregressively sample new tokens.

        Args:
            idx: (B, T) current context (token IDs)
            max_new_tokens: number of tokens to generate
            temperature: > 0, controls randomness (lower is more deterministic)

        Returns:
            idx: (B, T + max_new_tokens) with generated tokens appended
        """
        self.eval()

        for _ in range(max_new_tokens):

            # -------------------------------------------------------- #
            # --------------------[YOUR CODE HERE]-------------------- #
            # Keep the last block_size tokens to stay within the context window limit
            idx_max = idx[:, -block_size:]

            # Forward pass to get next token prediction for the current context
            logits, _ = self(idx_max)

            # Extract only the final row of predictions
            logits = logits[:, -1, :] # (B, T, vocab_size) -> (B, vocab_size)

            # Apply temperature to control randomness
            logits = logits / temperature

            # Apply row-wise softmax to convert raw logits into a probability distribution
            probs = F.softmax(logits, dim=-1) # (B, vocab_size)

            # Sample a token from the probability distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

            # Append the new token to the current sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T) -> (B, T+1)
            # -------------------------------------------------------- #




        return idx

## 8. Training specs

In [20]:
# Define hyperparameters (global variables). In a larger project, it is better
# practice to store these in a config object or a dataclass to avoid relying on globals.

# -------------------------------------------------------------------
# Choose the device we want to run on (CUDA > MPS > CPU)
# -------------------------------------------------------------------
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

# -------------------------------------------------------------------
# Training hyperparameters
# -------------------------------------------------------------------
batch_size = 16      # B: number of independent sequences (context windows) processed in parallel
block_size = 128     # T: maximum context length (number of tokens used to predict the next token)
max_iters = 5000     # total number of optimization steps
eval_interval = 500  # evaluate the model every eval_interval steps
eval_iters = 200     # number of batches used to estimate train/val loss during evaluation
learning_rate = 3e-4 # a common default for small GPT-style models (with AdamW)

# -------------------------------------------------------------------
# Model hyperparameters
# -------------------------------------------------------------------
n_embd = 384   # embedding dimension (must satisfy: n_head * head_size = n_embd)
n_head = 6     # number of attention heads (so head_size = n_embd // n_head = 64 here)
n_layer = 6    # number of decoder blocks
dropout = 0.2  # dropout probability (regularization)

# -------------------------------------------------------------------
# Build model and optimizer
# -------------------------------------------------------------------
model = GPTLanguageModel().to(device)

# Print the number of parameters (in millions)
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of model parameters: {num_params/1e6:.2f} million")

# AdamW is the standard optimizer for Transformer language models
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

Using device: cuda:0
Number of model parameters: 10.74 million


We now define a function to generate batches of sequences from our training or validation corpus. We will also define a function to obtain estimates of the loss function.

In [21]:
# ------------------------------------------------------------
# Mini DataLoader: sample random context windows from the corpus
# ------------------------------------------------------------
def get_batch(split: str):
    """
    Generate a batch of input (x) and target (y) sequences.

    Args:
        split: 'train' or 'val'

    Returns:
        x: (B, T) input token IDs
        y: (B, T) target token IDs (shifted by one position)
    """
    data_split = train_data if split == 'train' else val_data

    # Random starting positions for each sequence in the batch
    ix = torch.randint(len(data_split) - block_size, (batch_size,))

    # Build input and target sequences
    x = torch.stack([data_split[i:i + block_size] for i in ix])
    y = torch.stack([data_split[i + 1:i + block_size + 1] for i in ix])

    # Move to the selected device (CPU / GPU / MPS)
    x, y = x.to(device), y.to(device)

    return x, y


# ------------------------------------------------------------
# Estimate average loss on train and validation sets
# ------------------------------------------------------------
@torch.no_grad()
def estimate_loss():
    """
    Compute an average loss estimate over `eval_iters` batches
    for both training and validation splits.
    """
    out = {}
    model.eval()  # disable dropout, use evaluation mode

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()  # return to training mode
    return out

First sanity check: make sure the sampled batches have the correct dimensions.

In [22]:
# Verify that the sampled batches have the expected dimensions
xb, yb = get_batch('train')

print("Input batch shape :", xb.shape)  # Expected: (batch_size, block_size)
print("Target batch shape:", yb.shape)  # Expected: (batch_size, block_size)

Input batch shape : torch.Size([16, 128])
Target batch shape: torch.Size([16, 128])


Second sanity check: verify that the data flows correctly through the model.

In [23]:
# Verify that the model's forward pass runs without errors
# and produces logits of the expected shape

logits, _ = model(xb)

print("Logits shape:", logits.shape)
# Expected: (batch_size, block_size, vocab_size)

Logits shape: torch.Size([16, 128, 65])


## 9. Train

In [24]:
# Use your CID as the random seed (for reproducibility)
torch.manual_seed(2518058)

# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------
for step in range(max_iters):

    # Periodically estimate loss on both training and validation splits
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample a batch of training data
    xb, yb = get_batch('train')

    # Forward pass: compute logits and loss
    logits, loss = model(xb, yb)

    # Backward pass + parameter update
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.2672, val loss 4.2683
step 500: train loss 2.1804, val loss 2.2200
step 1000: train loss 1.7887, val loss 1.9088
step 1500: train loss 1.6194, val loss 1.7976
step 2000: train loss 1.5233, val loss 1.7112
step 2500: train loss 1.4794, val loss 1.6723
step 3000: train loss 1.4150, val loss 1.6066
step 3500: train loss 1.3838, val loss 1.5951
step 4000: train loss 1.3584, val loss 1.5800
step 4500: train loss 1.3404, val loss 1.5708
step 4999: train loss 1.3191, val loss 1.5593


## 10. Generate text sequences with the trained model

In [25]:
# ------------------------------------------------------------
# Generate text using the trained model
# ------------------------------------------------------------

# Start from a minimal context (single token with ID 0)
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# Generate 1500 new tokens autoregressively
generated_tokens = model.generate(context, max_new_tokens=1500)

# Decode token IDs back into characters
print(decode(generated_tokens[0].tolist()))


# ------------------------------------------------------------
# Optionally, write generated text to a file
# ------------------------------------------------------------

# open('output.txt', 'w').write(
#     decode(model.generate(context, max_new_tokens=10000)[0].tolist())
# )


Patrantorio in them flatter to love,
His hearts own facter to dight it.
Secary Paris and Duke of York,
You ouWhil on agreet: say you must one gracious vacardon,
Again the prince of the daughter Angelo.
Shalt this is but once sir, a curdery Barningbroke
My state are a king-looks.

CLAUDIO:
Come one mortage, and for likes, this rich is glory.

HENRY BOLINGBROKE:
Which in the otresty toath I may,
And not comfort to make the common-hath
And found my soul masterly backet's perform,
And that which I shall look old title lurder.

ESCALUS:
But, what kill'tis the curturant?

MERCUTIO:
My lord, by more stay lourney hath been course these
father than sent seeme that hasty world:
Whence the vengeance twere to be a forther'd o'erth,
Hath both thou made out admissed put
Behold o'erbe him, make more is than subject
Than serves the duke; am where the Visasits
Should detit on my.

CLAUDIO:
A morselusio, heir dears,
When, I fear not it,--pilp thy wife part by my perous.

NORFOLK:
O brief, sir, lie, for

In [28]:
generated_tokens_lowtemp = model.generate(context, max_new_tokens=1500, temperature=0.1)
print(decode(generated_tokens_lowtemp[0].tolist()))


Which is the prince of the prince of the country of the country.

CORIOLANUS:
What say you shall be so?

CORIOLANUS:
We have been to the state of the state of the state
Of the strength of the sent of the state of the state,
And the sent that the country of the sent of the sent
Of the prince of the sent that the prince of the sent
Of the sent that the sent that the sent of the state
Of the sentence of the sent that the sent of the sent
Of the sentence of the sent that the sent the state
Of the prince of the sent that the sent of the sent,
And the prince of the sent that the prince of the prince
Which they say the sent that the sent that the state
Of the prince of the prince of the seat,
And they say the sent that the strong that the sent of the sent
Of the strength of the prince of the sent
Of the sentence of the prince of the sent that the state,
And they shall be so so straight to the sent
Of the sent that the sent that the sent that the world.

First Servingman:
What say you shall b

In [29]:
generated_tokens_hightemp = model.generate(context, max_new_tokens=1500, temperature=2)
print(decode(generated_tokens_hightemp[0].tolist()))


Rouve?
'BriTl:
Cleforro high!
Wevewaotod: na?Sad, bu theyou loegeo; Ok from enoughing:
Fie! Bad'sLords say, b you: by you PraYherewn Hrcs carn.
Came!
You nose : btish'd quaked pparroaliance blucks how?
Off;Lock and, 'sucPLET:

CriUCIO:
Anmy duty,
As friar, much
there not leart useyN-cluh, girtum fermle:
Potr;for you lat; isfore telt,' Poli stillgome-daughter, friends,
Auch Lucivienc 'giv' otwo, for Thurcly. Heaw!-zaarew' bslain,-pardolute to Edwarr.
Wrst-ha? good unclighGy
Trbunesa busRievy!-pawsif Micklup.

VOLANO:
YORK, away, a.
'OrtuyS!

WadWage!
She:Cala! A fon's o't-kiss' a quaking it afzefnligazen;Vhe'tidan's
Is oboin; and like oned Kanety, Bur, he whets nig.

FRwARD:Y, ceeprah most king, Buckifkares an,
TRUnpregArly strugth yok:' I pumptor her behit;
Built to O brain my boociecas?
T'skxuls and yury;aa
MewC; Dy laquilius'd cambI, furzissh veult.yoatycondedith
Uvasslagor--beway fields!'Trtch'r'cloi! No, Voh,
lo!

USicOLEO:
Hath't inext Couyal orOx: jumbeas
M?

ANYEMAS

POMPEY:
Ye

### Questions
 - Q9 (20% marks): Comment on the generated text.
   - Does it learn local syntax?
   - Does it maintain long-term coherence?
   - How does the temperature affect the generated text?
   - What architectural limitation explains its behavior?

**Does it learn local syntax?**

Yes, the generated text shows that the model has learned the local syntax. It seems to capture basic grammar and many words look like a correct english words. The model also has proper spacing and full stop at the end of sentence. It also reproduces the format of the corpus. The text contains dialogue blocks that begin with a character’s name written in capital letters, followed by a colon and a newline.  

**Does it maintain long-term coherence?**

No, the model fails to maintain long-term coherence. While the local grammar is plausible, the overall sentence does not have meaning at all. For example, "Which in the otresty toath I may”, it literally just combine the english words without any meaning. Many words are also nonsensical like “morselusio”, “exhardishp” and there is also no logical relationship between each dialogue. 

**How does the temperature affect the generated text?**

Temperature controls the randomness of token sampling of the model's predictions by scaling the logits before the softmax function. 

- At low temperature (0.1), the model strongly favors the most probable tokens. As such, the text becomes very repetitive and deterministic, with many repeated phrases like “the prince of the sent of the sent”. The grammar structure is somewhat stable, but the output lacks creativity.
- At high temperature (2.0), the probability distribution is flattened, so the model samples much more randomly. This produces diverse but largely incoherent text, including many nonsensical words and broken syntax.

**What architectural limitation explains its behavior?**

One limitation is the fixed context length of `block_size = 128` . This means the model can only condition its predictions on the previous 128 characters. As a result, it cannot refer back to earlier parts of the dialogue to maintain long-term coherence. 

Another limitation is the small model size, with only ~10.74 million parameters. While this is sufficient to learn the local syntax and basic formatting, it does not provide enough modelling capacity to capture complex semantic relationships. 

Also, the model has a limited number of transformer layers and attention heads. As such, the network has less depth to process and refine representations of the text, and might struggle to attend multiple types of relationships between tokens.